In [ ]:
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import statistics
from tqdm import tqdm

In [ ]:
with open('/Webis-CPC-11_1000_sample.json', 'r') as file:
    data = json.load(file)

In [ ]:
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2', 
                            device = 0) # optional define GPU

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') # optional, define GPU
fin_res = []
for el in tqdm(data):
    
    # s-Text = original text
    # t_text - paraphrased text
    s_text = [dt['text'].strip() for dt in el.get('original')]
    t_text = [dt['text'].strip() for dt in el.get('paraphrase')]
    
    #Calculate similarity for the full texts
    
    tensor1 = model.encode(' '.join(s_text), convert_to_tensor=True)
    tensor2 = model.encode(' '.join(t_text), convert_to_tensor=True)
    
    tensor1 = tensor1.to(device)
    tensor2 = tensor2.to(device)
    
    sim = util.pytorch_cos_sim(tensor1, tensor2)
    sim_full = sim.item()
    
    dct = {'HITId' : el.get('HITId'),
           'WorkerId' : el.get('WorkerId'), 
           'is_paraphrased': el.get('is_paraphrased'),
           'full_sim' : sim_full
    }
    fin_res.append(dct)
fin_df = pd.DataFrame(fin_res)

In [ ]:
# show mean similarity 
fin_df[['is_paraphrased', 'full_sim']].groupby('is_paraphrased').mean()